In [1]:
from datasets import load_dataset
import torch
import numpy as np
import chess
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [2]:
dataset = load_dataset(
    "mateuszgrzyb/lichess-stockfish-normalized",
    split="train",
    cache_dir="./training_data"
)

Loading dataset shards:   0%|          | 0/35 [00:00<?, ?it/s]

In [3]:
split1 = dataset.train_test_split(test_size=0.2, seed=42)

train_ds = split1["train"]
temp_ds  = split1["test"]

# second split: val vs test
split2 = temp_ds.train_test_split(test_size=0.5, seed=42)

val_ds   = split2["train"]
test_ds  = split2["test"]

In [ ]:
#see https://official-stockfish.github.io/docs/nnue-pytorch-wiki/docs/nnue.html#halfkp
PIECE_MAP = {
    chess.PAWN: 0,
    chess.KNIGHT: 1,
    chess.BISHOP: 2,
    chess.ROOK: 3,
    chess.QUEEN: 4,
}

# using the same formula provided by the stockfish site
def halfkp_index(piece, piece_square, king_square):
    piece_type = PIECE_MAP[piece.piece_type]
    piece_color = 0 if piece.color == chess.WHITE else 1
    p_idx = piece_type * 2 + piece_color
    return piece_square + (p_idx + king_square * 10) * 64

def extract_halfkp(board, perspective):
    king_sq = board.king(perspective)
    feats = []

    for sq, piece in board.piece_map().items():
        if piece.piece_type == chess.KING:
            continue
        feats.append(halfkp_index(piece, sq, king_sq))

    return feats


In [ ]:
NUM_FEATURES = 40960 # 64*64*5*2 (Each feature is a tuple (our_king_square, piece_square, piece_type, piece_color))

class Accumulator:
    def __init__(self):
        self.white = torch.zeros(NUM_FEATURES, dtype=torch.int16)
        self.black = torch.zeros(NUM_FEATURES, dtype=torch.int16)

def init_accumulator(board):
    acc = Accumulator()

    w_king = board.king(chess.WHITE)
    b_king = board.king(chess.BLACK)

    for sq, piece in board.piece_map().items():
        if piece.piece_type == chess.KING:
            continue

        w_idx = halfkp_index(piece, sq, w_king)
        b_idx = halfkp_index(piece, sq, b_king)

        acc.white[w_idx] += 1
        acc.black[b_idx] += 1

    return acc

In [14]:
class ChessDataset(Dataset):
    def __init__(self, data, max_samples=None):
        self.ds = data

        if max_samples is not None:
            self.ds = self.ds.select(range(min(max_samples, len(self.ds))))

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        board = chess.Board(item["fen"])
        acc = init_accumulator(board)

        stm = 1.0 if board.turn == chess.WHITE else 0.0

        cp = item["cp"] or 0
        cp = max(min(cp, 1000), -1000) / 1000.0

        return {
            "white": acc.white,
            "black": acc.black,
            "stm": torch.tensor(stm, dtype=torch.float32),
            "target": torch.tensor([cp], dtype=torch.float32)
        }

In [11]:
def collate_fn(batch):
    white = torch.stack([b["white"] for b in batch]).float()
    black = torch.stack([b["black"] for b in batch]).float()
    stm = torch.stack([b["stm"] for b in batch])
    target = torch.stack([b["target"] for b in batch])

    return white, black, stm, target

In [10]:
class NNUE(nn.Module):
    def __init__(self, num_features=40960):
        super().__init__()

        self.white_fc = nn.Linear(num_features, 128)
        self.black_fc = nn.Linear(num_features, 128)

        # evaluation head
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 64)
        self.out = nn.Linear(64, 1)

        self.act = nn.ReLU()

    def forward(self, white, black, stm):
        # encode accumulator state
        w = self.act(self.white_fc(white))
        b = self.act(self.black_fc(black))

        x = torch.cat([w, b], dim=1)

        flip = (stm == 0).unsqueeze(1)
        x_flipped = torch.cat([b, w], dim=1)

        x = torch.where(flip, x_flipped, x)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.out(x)

In [38]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(device)

mps


In [39]:
BATCH_SIZE = 128
train_data = ChessDataset(train_ds, max_samples=100000)
val_data   = ChessDataset(val_ds, max_samples=20000)


train_loader = DataLoader(
    train_data,
    batch_size=128,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader = DataLoader(
    val_data,
    batch_size=128,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=True
)

In [40]:
model = NNUE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):

    # -------------------
    # TRAIN
    # -------------------
    model.train()
    train_loss = 0.0
    train_samples = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [TRAIN]")

    for white, black, stm, target in train_bar:
        batch_size = target.size(0)

        white = white.to(device).float()
        black = black.to(device).float()
        stm = stm.to(device)
        target = target.to(device)

        pred = model(white, black, stm)
        loss = loss_fn(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_size
        train_samples += batch_size

        train_bar.set_postfix(loss=loss.item())

    train_loss /= train_samples

    # -------------------
    # VALIDATION
    # -------------------
    model.eval()
    val_loss = 0.0
    val_samples = 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1} [VAL]")

    with torch.no_grad():
        for white, black, stm, target in val_bar:

            batch_size = target.size(0)

            white = white.to(device).float()
            black = black.to(device).float()
            stm = stm.to(device)
            target = target.to(device)

            pred = model(white, black, stm)
            loss = loss_fn(pred, target)

            val_loss += loss.item() * batch_size
            val_samples += batch_size

            val_bar.set_postfix(loss=loss.item())

    val_loss /= val_samples

    print(
        f"\nEpoch {epoch+1} Summary | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f}\n"
    )

Epoch 1 [VAL]: 100%|██████████| 157/157 [00:14<00:00, 11.18it/s, loss=0.0658]



Epoch 1 Summary | Train Loss: 0.070148 | Val Loss: 0.091152



Epoch 2 [VAL]: 100%|██████████| 157/157 [00:10<00:00, 14.35it/s, loss=0.0599]



Epoch 2 Summary | Train Loss: 0.042209 | Val Loss: 0.097616



Epoch 3 [VAL]: 100%|██████████| 157/157 [00:11<00:00, 13.93it/s, loss=0.0839]


...


Epoch 8 Summary | Train Loss: 0.007121 | Val Loss: 0.103046



Epoch 9 [VAL]: 100%|██████████| 157/157 [00:11<00:00, 13.84it/s, loss=0.118] 



Epoch 9 Summary | Train Loss: 0.006196 | Val Loss: 0.103273



Epoch 10 [VAL]: 100%|██████████| 157/157 [00:11<00:00, 13.93it/s, loss=0.101] 


Epoch 10 Summary | Train Loss: 0.005505 | Val Loss: 0.102147



In [ ]:
test_data  = ChessDataset(test_ds, max_samples=100000)
test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

model.eval()
test_loss = 0.0
test_mae = 0.0
test_samples = 0

test_bar = tqdm(test_loader, desc=f"[Test]")

with torch.no_grad():
    for white, black, stm, target in test_bar:

        batch_size = target.size(0)

        white = white.to(device).float()
        black = black.to(device).float()
        stm = stm.to(device)
        target = target.to(device)

        pred = model(white, black, stm)

        loss = loss_fn(pred, target)

        test_loss += loss.item() * batch_size
        test_mae += (pred - target).abs().sum().item()

        test_samples += batch_size

        test_bar.set_postfix(loss=loss.item())

# final averages
test_loss /= test_samples
test_mae /= test_samples

print(
    f"\n[Test Summary] "
    f"Loss: {test_loss:.6f} | "
    f"MAE: {test_mae:.6f}\n"
)

[Test]:   8%|▊         | 130/1563 [00:07<01:20, 17.91it/s, loss=0.0936]